In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [37]:
workdir = '../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/'
infile = workdir + '12B1_500kb.all_samples_counts.tab'    

In [38]:
chromdict = {'CM083856.1': 'scaf1',
'CM083857.1': 'scaf2',
'CM083858.1': 'scaf3',
'CM083859.1': 'scaf4',
'CM083860.1': 'scaf5',
'CM083861.1': 'scaf6',
'CM083862.1': 'scaf7',
'CM083863.1': 'scaf8',
'CM083864.1': 'scaf9',
'CM083865.1': 'scaf10',
'CM083866.1': 'scaf11',
'CM083867.1': 'scaf12',
'CM083868.1': 'scaf13',
'CM083869.1': 'scaf14',
'CM083870.1': 'scaf15',
'CM083871.1': 'scaf16',
'CM083872.1': 'scaf17',
'CM083873.1': 'scaf18',
'CM083874.1': 'scaf19',
'CM083875.1': 'scaf20',
'CM083876.1': 'scaf21',
'CM083877.1': 'scaf22',
'CM083878.1': 'scaf23',
'CM083879.1': 'scaf24',
'CM083880.1': 'scaf25',
'CM083881.1': 'scaf26',
'CM083882.1': 'scaf27',
'CM083883.1': 'scaf28',
'CM083884.1': 'scaf29',
'CM083885.1': 'scaf30',
'CM083886.1': 'scaf31',
'CM083887.1': 'scaf32',
'CM083888.1': 'scaf33',
'CM083889.1': 'scaf34'}

In [39]:
# Load raw counts
counts = pd.read_csv(infile, sep="\t")
counts.columns = counts.columns.str.strip("'#").str.strip()

In [40]:
# Remove chloroplast scaffold
counts = counts[counts["chr"] != "scf7180000001165"]

# Remove small scaffolds 31-34
#counts = counts[counts["chr"] != "CM083885.1"]
#counts = counts[counts["chr"] != "CM083886.1"]
counts = counts[counts["chr"] != "CM083887.1"]
counts = counts[counts["chr"] != "CM083888.1"]
counts = counts[counts["chr"] != "CM083889.1"]


In [41]:
# Rename chromosomes 
counts["chr"] = counts["chr"].replace(chromdict)
all_samples = counts.columns[3:].tolist()
counts

,chr,start,end,12A1,12A1-SC1,12A1-SC2,12A1-SC3,12B1,13A5,8020,...,PK2298C6,PK2299B3,PK2300D3,RCC3703,UTEX2797,UTEX2797-SC1,UTEX2797-SC2,UTEX2797-SC3,UTEX2797-SC4,UTEX2827
0,scaf1,0,500000,3298.889893,3331.100098,3255.530029,3285.010010,3293.560059,3238.800049,3295.610107,...,3350.689941,3206.000000,3326.580078,3346.719971,3187.639893,3160.780029,3236.959961,3179.919922,3283.610107,3642.419922
1,scaf1,500000,1000000,3467.800049,3530.760010,3461.320068,3417.489990,3511.649902,3500.040039,3598.810059,...,3647.100098,3629.080078,3709.709961,3780.280029,3771.239990,3898.000000,3612.060059,3668.149902,3561.709961,3371.860107
2,scaf1,1000000,1500000,3403.439941,3471.280029,3374.590088,3365.840088,3437.409912,3399.389893,3549.770020,...,3493.510010,3551.409912,3526.040039,3563.080078,3660.479980,3829.250000,3507.590088,3566.040039,3542.669922,3649.370117
3,scaf1,1500000,2000000,3507.139893,3584.530029,3505.100098,3479.179932,3693.979980,3525.780029,3640.820068,...,3615.679932,3703.860107,3628.310059,3549.389893,3772.850098,3973.209961,3593.510010,3665.330078,3613.929932,4714.339844
4,scaf1,2000000,2500000,3146.060059,3222.389893,3179.060059,3111.909912,3270.000000,3196.419922,3293.050049,...,3304.600098,3383.899902,3360.250000,3964.860107,3481.399902,3621.540039,3253.790039,3341.050049,3234.479980,4332.009766
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,scaf30,500000,1000000,3366.689941,3377.379883,3318.959961,3389.689941,3348.290039,3347.939941,3451.989990,...,3444.149902,3197.870117,3491.290039,3496.879883,3242.250000,3265.520020,3310.449951,3251.090088,3332.260010,3186.030029
194,scaf30,1000000,1325017,1739.500000,1757.270020,1724.640015,1784.199951,1782.670044,1711.290039,1825.650024,...,1675.180054,1568.109985,1666.089966,1700.719971,1609.680054,1537.670044,1696.780029,1589.800049,1734.689941,1489.790039
195,scaf31,0,500000,2647.550049,2633.439941,2589.610107,2756.679932,2839.600098,2609.979980,2625.500000,...,2814.780029,2410.429932,2771.290039,2583.949951,3255.370117,2556.139893,2838.080078,3604.679932,3815.159912,2323.879883
196,scaf31,500000,1000000,2635.090088,2622.620117,2590.520020,2692.590088,2650.350098,2573.139893,2727.550049,...,2727.110107,2304.010010,2695.870117,2787.320068,3331.020020,2332.659912,2631.300049,3549.520020,3797.889893,2359.050049


In [46]:
# ── Within-sample normalization ───────────────────────────────────────────────
# Divide each bin by the sample's own genome-wide median
# This removes sequencing depth differences between samples
# while preserving relative chromosomal imbalance within each sample
sample_data = counts[all_samples]
within_normalized = sample_data.div(sample_data.median(axis=0), axis=1)

norm_df = counts[["chr", "start", "end"]].copy()
norm_df = pd.concat([norm_df, within_normalized], axis=1)
norm_df

,chr,start,end,12A1,12A1-SC1,12A1-SC2,12A1-SC3,12B1,13A5,8020,...,PK2298C6,PK2299B3,PK2300D3,RCC3703,UTEX2797,UTEX2797-SC1,UTEX2797-SC2,UTEX2797-SC3,UTEX2797-SC4,UTEX2827
0,scaf1,0,500000,1.009143,1.005732,0.993890,1.004578,1.001659,0.991960,0.986166,...,1.000224,0.981887,0.988560,0.995323,0.980648,0.948019,0.989094,0.968648,1.006563,1.142105
1,scaf1,500000,1000000,1.060814,1.066014,1.056716,1.045091,1.067986,1.071971,1.076895,...,1.088706,1.111462,1.102415,1.124265,1.160187,1.169135,1.103711,1.117369,1.091812,1.057270
2,scaf1,1000000,1500000,1.041126,1.048055,1.030238,1.029296,1.045408,1.041145,1.062220,...,1.042858,1.087675,1.047834,1.059669,1.126113,1.148514,1.071789,1.086265,1.085976,1.144285
3,scaf1,1500000,2000000,1.072848,1.082248,1.070082,1.063956,1.123437,1.079855,1.089466,...,1.079327,1.134365,1.078226,1.055598,1.160682,1.191692,1.098043,1.116510,1.107820,1.478213
4,scaf1,2000000,2500000,0.962392,0.972910,0.970544,0.951643,0.994494,0.978980,0.985400,...,0.986465,1.036372,0.998566,1.179160,1.071020,1.086215,0.994237,1.017730,0.991503,1.358331
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,scaf30,500000,1000000,1.029884,1.019705,1.013254,1.036590,1.018304,1.025387,1.032961,...,1.028123,0.979398,1.037507,1.039981,0.997448,0.979434,1.011550,0.990327,1.021476,0.999001
194,scaf30,1000000,1325017,0.532120,0.530558,0.526520,0.545620,0.542157,0.524124,0.546301,...,0.500063,0.480258,0.495112,0.505799,0.495203,0.461196,0.518473,0.484275,0.531755,0.467134
195,scaf31,0,500000,0.809896,0.795093,0.790589,0.843011,0.863598,0.799369,0.785645,...,0.840248,0.738232,0.823545,0.768474,1.001484,0.766668,0.867212,1.098035,1.169505,0.728668
196,scaf31,500000,1000000,0.806084,0.791826,0.790867,0.823412,0.806042,0.788086,0.816182,...,0.814077,0.705639,0.801132,0.828956,1.024757,0.699639,0.804027,1.081233,1.164211,0.739696


In [43]:
# ── Per-chromosome median (within-normalized) ─────────────────────────────────
chr_summary = (
    norm_df
    .groupby("chr")[all_samples]
    .median()
)

# Natural chromosome sort
chr_summary = chr_summary.loc[
    sorted(chr_summary.index,
           key=lambda x: (0, int(x.replace("scaf","").split("_")[0]))
                          if x.replace("scaf","").split("_")[0].isdigit()
                          else (1, x))
]
chr_summary

,12A1,12A1-SC1,12A1-SC2,12A1-SC3,12B1,13A5,8020,8027,ARC66,ARC81,...,PK2298C6,PK2299B3,PK2300D3,RCC3703,UTEX2797,UTEX2797-SC1,UTEX2797-SC2,UTEX2797-SC3,UTEX2797-SC4,UTEX2827
chr,,,,,,,,,,,,,,,,,,,,,
scaf1,1.058868,1.059507,1.042077,1.045091,1.045408,1.061264,1.065640,1.027703,1.080968,1.084900,...,1.059518,1.111043,1.067290,1.124265,1.126113,1.148514,1.071789,1.088032,1.085976,1.140453
scaf2,1.016136,0.960217,1.018298,0.999379,1.010257,1.031081,1.037599,1.026384,1.040372,1.073511,...,1.039795,1.020002,1.035320,1.058965,1.056435,1.110093,1.035390,1.053665,1.026684,1.000002
scaf3,0.988247,0.979365,0.961443,0.986707,0.963400,0.976178,0.996972,0.975690,1.010494,0.980610,...,0.994486,0.999384,0.982026,0.983478,0.985398,1.014802,0.990845,0.982075,1.010686,0.999970
scaf4,1.036655,1.044485,1.041430,1.041808,1.026873,1.038445,1.033393,1.039121,1.057189,1.032100,...,1.034598,1.023811,1.043812,1.045760,1.047349,1.047445,1.034941,1.027365,1.081234,1.031686
scaf5,1.007322,1.020094,1.011728,0.997237,1.012383,1.026333,1.027042,1.000398,1.025635,1.059949,...,1.013876,1.054915,1.033485,1.027092,1.093978,1.101219,1.031841,1.055315,1.030084,1.067874
scaf6,0.966273,0.971304,0.959988,0.968941,0.979438,0.958938,0.974159,0.986598,1.017217,1.014548,...,1.000410,0.996708,1.000808,0.958750,1.017459,1.022669,1.007919,1.007146,0.994534,1.051406
scaf7,1.020913,1.035616,1.025345,1.026668,1.039193,1.026065,1.049269,1.044579,1.075209,1.109649,...,1.054838,1.063356,1.061810,1.090592,1.086495,1.098243,1.044476,1.046007,1.052658,1.076539
scaf8,1.317092,0.942550,1.177316,1.313848,0.958946,1.328568,0.969297,1.037769,0.971718,0.953720,...,1.073736,1.328791,0.952157,0.860115,0.944686,0.975578,1.037926,1.029351,0.950779,0.932251
scaf9,1.060473,1.054714,1.044655,1.055616,1.061545,1.056370,1.071637,1.048988,1.079478,1.068215,...,1.066243,1.027218,1.071206,1.160531,1.438891,1.045981,1.438851,1.408154,1.537798,1.536505


In [44]:
# ── Score each sample by chromosomal coverage uniformity ─────────────────────
# Several complementary metrics — use together rather than any single one

uniformity = pd.DataFrame(index=all_samples)

# 1. Coefficient of variation across chromosomes
#    Lower = more uniform = more likely euploid
uniformity["cv"] = chr_summary.std() / chr_summary.mean()

# 2. IQR of per-chromosome medians (robust to outlier chromosomes)
uniformity["iqr"] = chr_summary.apply(lambda x: stats.iqr(x))

# 3. Max deviation from 1.0 (within-normalized, so euploid expectation is 1.0)
uniformity["max_deviation"] = (chr_summary - 1.0).abs().max()

# 4. Number of chromosomes outside a loose aneuploidy window
#    Equivalent to log2 ratio threshold of ~0.2 in relative terms
uniformity["n_outlier_chr"] = (
    (chr_summary - 1.0).abs() > 0.2
).sum()

# Rank by CV as primary metric, use others to corroborate
uniformity = uniformity.sort_values("cv")

print(uniformity.round(4).to_string())
#uniformity.to_csv("coverage_uniformity_scores.tsv", sep="\t")

                  cv     iqr  max_deviation  n_outlier_chr
CC2192D3      0.0810  0.0818         0.2839              1
CC2192B5      0.0820  0.1055         0.2729              1
8027          0.0917  0.1163         0.3214              2
CC2192C1      0.0950  0.0934         0.3155              3
PK2298B4      0.0978  0.1183         0.3157              3
CCMP3037      0.1024  0.0956         0.3232              3
PK2298C6      0.1035  0.1033         0.3096              3
12A1          0.1125  0.0774         0.3171              2
12A1-SC2      0.1127  0.1055         0.3279              3
CC2192B4      0.1128  0.1088         0.3484              4
8020          0.1160  0.1159         0.3997              3
ARC82         0.1163  0.1002         0.3966              4
PK2300D3      0.1175  0.1308         0.3761              3
12A1-SC1      0.1189  0.0950         0.3767              3
12A1-SC3      0.1252  0.0865         0.3671              3
CC2192D2      0.1287  0.1287         0.3904             

In [12]:
# ── Identify candidate references ─────────────────────────────────────────────
# Look for a natural break in the CV distribution
# rather than applying a hard threshold
cv_sorted = uniformity["cv"].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# CV distribution — look for a gap separating uniform from variable samples
axes[0].scatter(range(len(cv_sorted)), cv_sorted, c='steelblue', s=40)
axes[0].set_xlabel("Sample rank")
axes[0].set_ylabel("CV of per-chromosome median coverage")
axes[0].set_title("Coverage uniformity ranking\n(lower = more euploid-like)")
for i, (sample, row) in enumerate(uniformity.iterrows()):
    axes[0].annotate(sample, (i, row["cv"]), fontsize=6,
                     rotation=45, ha='left', va='bottom')

# Heatmap of per-chromosome coverage for all samples, ordered by CV
axes[1].set_visible(False)
fig.tight_layout()
plt.savefig("2_uniformity_ranking.pdf", bbox_inches="tight")
plt.close()

# Full heatmap separately
fig2, ax = plt.subplots(figsize=(14, 8))
heatmap_data = chr_summary[uniformity.index].T  # samples ordered by CV
sns.heatmap(
    heatmap_data,
    center=1.0,
    cmap="RdBu_r",
    vmin=0.5, vmax=1.5,
    ax=ax,
    xticklabels=True,
    yticklabels=True
)
ax.set_title("Per-chromosome median coverage (within-sample normalized)\nSamples ordered by coverage uniformity")
ax.set_xlabel("Chromosome")
ax.set_ylabel("Sample (ordered by CV, top = most uniform)")
plt.tight_layout()
plt.savefig("2_chr_coverage_heatmap_ranked.pdf", bbox_inches="tight")
plt.close()

print("\nTop candidate reference strains:")
print(uniformity.head(8).round(4).to_string())


Top candidate reference strains:
              cv     iqr  max_deviation  n_outlier_chr
CC2192D3  0.0406  0.0359         0.1143              0
CC2192B5  0.0443  0.0513         0.1217              0
CC2192C1  0.0485  0.0595         0.1353              0
CCMP3037  0.0525  0.0789         0.1429              0
8020      0.0585  0.0799         0.1325              0
PK2300D3  0.0595  0.0668         0.1542              0
PK2298C6  0.0626  0.0639         0.1586              0
CC2192B4  0.0658  0.0746         0.1757              0


In [45]:
# ── Identify which chromosome is the outlier in candidate references ──────────
candidates = ["CC2192B5", "CC2192D3", "8027"]

print("Outlier chromosomes in candidate reference strains:")
for sample in candidates:
    outliers = chr_summary.index[
        (chr_summary[sample] - 1.0).abs() > 0.2
    ].tolist()
    vals = chr_summary.loc[outliers, sample].round(3).to_dict()
    print(f"  {sample}: {vals}")

Outlier chromosomes in candidate reference strains:
  CC2192B5: {'scaf30': 0.727}
  CC2192D3: {'scaf30': 0.716}
  8027: {'scaf30': 0.679, 'scaf31': 0.781}


In [15]:
#Decided to use CC2192B5 and CC2192D3 as the two references 
#excluding scaffolds 30-34, which resulted in CC2192B5 and CC2192D3
#having no outlier chromosomes regardless of binsize used (10kb - 500kb)